# Week 70, Day 5: Advanced exam

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w070d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: The timer keeps running if you reload or close the page. It doesn't stop you working when time is up: that's on your honour..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: B1: Taking turns

Write a generator `interleave(*iterables)` that yields one item from each iterable in turn, skipping any that are used up, until all are (`interleave("abc", [1], "xy")` gives `a`, `1`, `x`, `b`, `y`, `c`). It must be lazy: it works when some of the iterables are infinite, and it takes an item from an iterable only when that item is about to be yielded.

In [ ]:
def interleave(*iterables):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_interleave():\n    \"round robin\"\n    import itertools\n    assert list(interleave(\"abc\", [1], \"xy\")) == [\"a\", 1, \"x\", \"b\", \"y\", \"c\"]\n    assert list(interleave()) == [] and list(interleave([], [])) == []\n    assert list(itertools.islice(interleave(itertools.count(), \"ab\"), 6)) == [0, \"a\", 1, \"b\", 2, 3]\n    pulled = []\n    def src():\n        for i in range(10):\n            pulled.append(i)\n            yield i\n    g = interleave(src(), \"pq\")\n    assert [next(g), next(g), next(g)] == [0, \"p\", 1] and pulled == [0, 1]")

### Exercise: B2: Only once

Write a decorator `once(func)`: the first call runs `func` and remembers its result; later calls return that result without running `func` again, whatever their arguments. If the first call raises, nothing is remembered, so the next call tries again. Keep the function's name and docstring, and give the wrapper a `reset()` method (an attribute that's a function) that forgets the result so the next call runs `func` again.

In [ ]:
import functools

def once(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_once():\n    \"a remembered first call\"\n    calls = []\n    @once\n    def setup(x):\n        \"Set things up.\"\n        calls.append(x)\n        return x * 10\n    assert setup(1) == 10 and setup(2) == 10 and calls == [1]\n    assert setup.__name__ == \"setup\" and setup.__doc__ == \"Set things up.\"\n    setup.reset()\n    assert setup(3) == 30 and calls == [1, 3]\n    attempts = []\n    @once\n    def flaky():\n        attempts.append(1)\n        if len(attempts) == 1:\n            raise OSError\n        return \"ok\"\n    try:\n        flaky()\n    except OSError:\n        pass\n    assert flaky() == \"ok\" and flaky() == \"ok\" and len(attempts) == 2\n    @once\n    def nothing():\n        attempts.append(\"n\")\n        return None\n    nothing(); nothing()\n    assert attempts.count(\"n\") == 1")

### Exercise: B3: Predict

Fill in each value without running anything.

In [ ]:
p = ___   # fs = [lambda: i for i in range(3)]; [f() for f in fs]
q = ___   # it = iter(range(6)); list(zip(it, it))
r = ___   # a = [1, 2]; b = a[:]; b.append(3); len(a)
s = ___   # class A: x = 1;  a = A(); a.x = 2; del a.x; a.x
t = ___   # True or False: asyncio.sleep(1) called inside a coroutine without await pauses it

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [2, 2, 2] and q == [(0, 1), (2, 3), (4, 5)] and r == 2, \"p, q and r\"\n    assert s == 1 and t is False, \"s and t\"")

### Exercise: B4: Nested timers

Write a class `Timings(clock)`, where `clock()` returns the current time. Its method `section(name)` is a context manager: it times its block with `clock` and adds the elapsed time to `self.totals`, a dict keyed by the section's **path**: its name joined with the names of the sections it's nested inside, using `/` (so `section("parse")` inside `section("load")` is `"load/parse"`). Repeated sections add up. Time is recorded, and the nesting restored, even when the block raises (the exception propagates). `report()` returns the `(path, total)` pairs sorted by path.

In [ ]:
from contextlib import contextmanager

class Timings:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timings():\n    \"nested context managers\"\n    now = [0]\n    t = Timings(lambda: now[0])\n    with t.section(\"load\"):\n        now[0] = 1\n        with t.section(\"parse\"):\n            now[0] = 4\n        with t.section(\"parse\"):\n            now[0] = 6\n    with t.section(\"save\"):\n        now[0] = 7\n    assert t.totals == {\"load/parse\": 5, \"load\": 6, \"save\": 1}\n    try:\n        with t.section(\"load\"):\n            now[0] = 9\n            raise ValueError\n    except ValueError:\n        pass\n    with t.section(\"end\"):\n        now[0] = 10\n    assert t.report() == [(\"end\", 1), (\"load\", 8), (\"load/parse\", 5), (\"save\", 1)]")

### Exercise: B5: Counting references

Write `aliases(obj)`: walk `obj` and everything inside it (list and tuple items, and dict **values**) and return `(distinct, repeated)`: how many distinct list, tuple and dict objects there are (compared by identity, not equality), and how many times the walk reached a container it had already seen. Containers inside a cycle must not make it loop forever. Other objects (numbers, strings and so on) aren't counted.

In [ ]:
def aliases(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_aliases():\n    \"identity, sharing and cycles\"\n    shared = [1]\n    assert aliases([shared, shared, (shared,), {\"k\": shared}]) == (4, 3)\n    assert aliases([[1], [1]]) == (3, 0) and aliases(5) == (0, 0)\n    loop = []\n    loop.append(loop)\n    assert aliases(loop) == (1, 1)\n    d = {\"a\": {\"b\": [2]}}\n    d[\"a\"][\"self\"] = d\n    assert aliases(d) == (3, 1)")

### Exercise: B6: Layered settings

Write a class `Settings(parent=None, **values)`. Reading an attribute returns this layer's value if it has one, otherwise the parent's (and so on up the chain); a name no layer has raises `AttributeError` with the message `"no setting 'name'"`. Settings are read-only: assigning or deleting any attribute raises `AttributeError("settings are read-only")`. `override(**values)` returns a new layer on top of this one, and `to_dict()` returns every visible setting, with nearer layers winning.

In [ ]:
class Settings:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"__getattr__ and __setattr__\"\n    base = Settings(colour=\"blue\", size=10)\n    mine = base.override(size=12)\n    assert (mine.colour, mine.size, base.size) == (\"blue\", 12, 10)\n    assert mine.to_dict() == {\"colour\": \"blue\", \"size\": 12} and base.to_dict() == {\"colour\": \"blue\", \"size\": 10}\n    try:\n        mine.font\n        assert False\n    except AttributeError as e:\n        assert str(e) == \"no setting 'font'\"\n    for action in (lambda: setattr(mine, \"size\", 1), lambda: delattr(mine, \"size\")):\n        try:\n            action()\n            assert False\n        except AttributeError as e:\n            assert str(e) == \"settings are read-only\"\n    assert mine.size == 12 and not hasattr(base, \"font\")\n    import copy\n    assert copy.copy(mine).size == 12")

### Exercise: B7: Modules from strings

Write `install(sources)`: `sources` is a dict of module name to source code. Put a finder at the **front** of `sys.meta_path` so that `import name` (and `importlib.import_module(name)`) loads those modules from their source, and return the finder. The finder records the names of the modules it has loaded in a list `finder.loaded`, in load order. Then `uninstall(finder)`: remove it from `sys.meta_path` and remove every module it loaded from `sys.modules`. Other imports must be unaffected.

In [ ]:
import sys
import importlib.abc
import importlib.util

def install(sources):
    ___

def uninstall(finder):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_install():\n    \"finders, specs and loaders\"\n    import importlib, sys\n    finder = install({\"alpha_mod\": \"VALUE = 41\", \"beta_mod\": \"import alpha_mod\\nVALUE = alpha_mod.VALUE + 1\"})\n    try:\n        assert sys.meta_path[0] is finder\n        beta = importlib.import_module(\"beta_mod\")\n        assert beta.VALUE == 42 and finder.loaded == [\"beta_mod\", \"alpha_mod\"]\n        assert importlib.import_module(\"alpha_mod\") is sys.modules[\"alpha_mod\"] and finder.loaded == [\"beta_mod\", \"alpha_mod\"]\n        import json\n        assert json.dumps([1]) == \"[1]\"\n    finally:\n        uninstall(finder)\n    assert finder not in sys.meta_path and \"alpha_mod\" not in sys.modules and \"beta_mod\" not in sys.modules\n    try:\n        importlib.import_module(\"alpha_mod\")\n        assert False\n    except ImportError:\n        pass")

### Exercise: B8: Gathering with a limit

Coroutines here are generators: each `yield` hands the loop a number of seconds to sleep, and the `return` value is the result. Write `gather_limited(coros, limit)`: run the coroutines in the list with at most `limit` running at once. The first `limit` start at time 0, in list order; each time one finishes, the next unstarted one starts at that moment. The loop always resumes the task that wakes earliest (ties in the order they went to sleep or started). Return `(results, total)`: the results **in the list's order** and the time the last one finished (0 for an empty list).

In [ ]:
import heapq

def gather_limited(coros, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_gather():\n    \"a semaphore, simulated\"\n    def job(waits, result):\n        for w in waits:\n            yield w\n        return result\n    def jobs():\n        return [job([3], \"a\"), job([1, 1], \"b\"), job([1], \"c\")]\n    assert gather_limited(jobs(), 2) == ([\"a\", \"b\", \"c\"], 3)\n    assert gather_limited(jobs(), 1) == ([\"a\", \"b\", \"c\"], 6)\n    assert gather_limited(jobs(), 5) == ([\"a\", \"b\", \"c\"], 3)\n    assert gather_limited([], 2) == ([], 0)\n    order = []\n    def noisy(name, waits):\n        order.append((\"start\", name))\n        for w in waits:\n            yield w\n        order.append((\"end\", name))\n        return name\n    gather_limited([noisy(\"x\", [2]), noisy(\"y\", [1]), noisy(\"z\", [1])], 2)\n    assert order == [(\"start\", \"x\"), (\"start\", \"y\"), (\"end\", \"y\"), (\"start\", \"z\"), (\"end\", \"x\"), (\"end\", \"z\")]")

### Exercise: C: A build tool

Write a class `Builder` for describing and running build steps.

- **`rule(name, deps=(), cost=1)`** is a decorator that registers the decorated function as the rule `name`, which depends on the rules named in `deps` and takes `cost` seconds; it returns the function unchanged. Registering a name twice raises `ValueError`.
- **`order(target)`** returns the names of the rules `target` needs, itself included, each once, with every rule after its dependencies: visit each rule's `deps` in the order listed, depth first, adding a rule after its dependencies. An unknown rule raises `KeyError`; a cycle raises `ValueError` whose message starts `"cycle: "` and lists the path, such as `"cycle: x -> y -> x"`.
- **`build(target)`** runs the rules in `order(target)` that don't have a cached result, each called with the results of its `deps` as positional arguments, in order; it caches every result and returns `target`'s.
- **`invalidate(name)`** forgets the cached results of `name` and of every rule that depends on it, directly or indirectly.
- **`dry_run()`** is a context manager yielding a list: inside the block, `build` calls nothing and caches nothing, appends to that list the names it **would** run, and returns `None`. Normal building resumes when the block ends, even after an error.
- **`timeline(target, workers)`** simulates building everything in `order(target)` (ignoring the cache) on `workers` workers: a rule can start once all its dependencies have finished; whenever a worker is free it takes the ready rule that comes **earliest** in `order(target)`. Return `(starts, makespan)`: a dict of each rule's start time, and the time the last one finished.

In [ ]:
import heapq
from contextlib import contextmanager

class Builder:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make():\n    b = Builder()\n    calls = []\n    @b.rule(\"config\")\n    def config():\n        calls.append(\"config\")\n        return {\"debug\": False}\n    @b.rule(\"lib\", deps=[\"config\"], cost=3)\n    def lib(cfg):\n        calls.append(\"lib\")\n        return \"lib-dbg\" if cfg[\"debug\"] else \"lib\"\n    @b.rule(\"docs\", deps=[\"config\"], cost=2)\n    def docs(cfg):\n        calls.append(\"docs\")\n        return \"docs\"\n    @b.rule(\"app\", deps=[\"lib\", \"config\"], cost=2)\n    def app(l, cfg):\n        calls.append(\"app\")\n        return f\"app({l})\"\n    @b.rule(\"all\", deps=[\"app\", \"docs\"])\n    def everything(a, d):\n        calls.append(\"all\")\n        return [a, d]\n    return b, calls, everything\n\ndef test_rules_and_order():\n    \"registering and ordering\"\n    b, calls, everything = make()\n    assert everything.__name__ == \"everything\" and calls == []\n    assert b.order(\"all\") == [\"config\", \"lib\", \"app\", \"docs\", \"all\"] and b.order(\"lib\") == [\"config\", \"lib\"]\n    try:\n        b.rule(\"lib\")(lambda: 0)\n        assert False\n    except ValueError:\n        pass\n    c = Builder()\n    c.rule(\"x\", deps=[\"y\"])(lambda y: y)\n    c.rule(\"y\", deps=[\"x\"])(lambda x: x)\n    c.rule(\"z\", deps=[\"nowhere\"])(lambda n: n)\n    try:\n        c.order(\"x\")\n        assert False\n    except ValueError as e:\n        assert str(e) == \"cycle: x -> y -> x\"\n    try:\n        c.order(\"z\")\n        assert False\n    except KeyError:\n        pass\n\ndef test_build_and_cache():\n    \"building, caching and invalidating\"\n    b, calls, _ = make()\n    assert b.build(\"all\") == [\"app(lib)\", \"docs\"] and calls == [\"config\", \"lib\", \"app\", \"docs\", \"all\"]\n    assert b.build(\"all\") == [\"app(lib)\", \"docs\"] and len(calls) == 5\n    b.invalidate(\"lib\")\n    assert b.build(\"all\") == [\"app(lib)\", \"docs\"] and calls[5:] == [\"lib\", \"app\", \"all\"]\n    b.invalidate(\"config\")\n    assert b.build(\"lib\") == \"lib\" and calls[8:] == [\"config\", \"lib\"]\n\ndef test_dry_run():\n    \"a context manager that changes behaviour\"\n    b, calls, _ = make()\n    b.build(\"lib\")\n    with b.dry_run() as planned:\n        assert b.build(\"all\") is None\n    assert planned == [\"app\", \"docs\", \"all\"] and calls == [\"config\", \"lib\"]\n    try:\n        with b.dry_run():\n            raise KeyError\n    except KeyError:\n        pass\n    assert b.build(\"all\") == [\"app(lib)\", \"docs\"] and calls[2:] == [\"app\", \"docs\", \"all\"]\n\ndef test_timeline():\n    \"a simulated parallel build\"\n    b, calls, _ = make()\n    assert b.timeline(\"all\", 2) == ({\"config\": 0, \"lib\": 1, \"docs\": 1, \"app\": 4, \"all\": 6}, 7)\n    assert b.timeline(\"all\", 1) == ({\"config\": 0, \"lib\": 1, \"app\": 4, \"docs\": 6, \"all\": 8}, 9)\n    assert b.timeline(\"lib\", 3) == ({\"config\": 0, \"lib\": 1}, 4) and calls == []")

## Before you start

**The rules.** You have **2 hours and 50 minutes** in total. Work alone. Close your notes, don't use the tutor, and don't search the web: the point is to find out what **you** can do, and the debrief can only help with gaps it can see. You may run your code and use **Check my answer** as often as you like.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 20 quiz questions | 20 (1 each) |
| B | 8 problems | 110 |
| C | one larger program | 50 |
| | **Total** | **180** |

Count a problem's full marks only when every check passes. **108 (60%)** passes Phase 5; **144 (80%)** is a distinction. If you don't pass, tomorrow's debrief tells you what to revise, and you can take the exam again after the rest week.

**Strategy.** Read every problem first and note which tool each needs. Do the ones you're sure of first, and don't stay stuck on one for more than 20 minutes. Test small and awkward cases (empty input, errors in the middle, a second run) before pressing Check.

## Part A: quiz

## Part B: eight problems

## Part C: a larger program

## When you finish

Add up your marks: 1 for each quiz question right first time, and the full marks for each problem whose checks all pass. Write your total down, with the time you took and a note of every problem you didn't finish or found hard. Don't look anything up: tomorrow's debrief goes through everything, with a second chance at each skill.